# Regime Module Foundation — Hypothesis Validation

**Goal**: Validate the regime module's behavior on real data before building the Regime-Weighted Ensemble.

| # | Hypothesis | Purpose |
|---|-----------|----------|
| H1 | Regime Distribution & Transition Frequency | Enough transitions per group to calibrate weights? |
| H2 | Conditional Returns Differ by Regime | Alpha in conditioning on regime? (Welch t-test) |
| H3 | Posterior Continuity | p_trending / vol_percentile smooth enough for soft blending? |
| H4 | 9→4 Grouping Validity | Is TREND_BULL / TREND_BEAR / RANGE / CHOPPY the right split? |
| H5 | MTF Alignment Value | Does HTF confirmation improve forward returns? |

**Data**: Binance BTCUSDT + ETHUSDT, 1h + 4h, 6 months (Dec 2025 – May 2026)

In [ ]:
import sys, types
sys.path.insert(0, '../src')

# Create 'app' namespace alias (regime module uses app.regime.* imports)
import importlib
app = types.ModuleType('app')
app.__path__ = ['../src/libs', '../src/apps']
sys.modules['app'] = app

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from datetime import datetime, timezone
from scipy import stats as sp_stats
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from binance.um_futures import UMFutures

# Regime module imports
from libs.regime.orchestrator import RegimeOrchestrator
from libs.regime.aggregation.rule_based import (
    ALL_REGIMES, TREND_REGIMES, NON_TREND_REGIMES,
    BULL_REGIMES, BEAR_REGIMES, CHOPPY,
    CLEAN_TREND_BULL, CLEAN_TREND_BEAR,
    VOLATILE_TREND_BULL, VOLATILE_TREND_BEAR,
    QUIET_MR_RANGE, QUIET_MR_SQUEEZE,
)

plt.style.use('dark_background')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

client = UMFutures()
print('Setup complete')

## Data Fetch — OHLCV (Binance Futures)

In [ ]:
# ── OHLCV fetch with pagination ────────────────────────────────────
_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'
]

def fetch_ohlcv(symbol: str, interval: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch OHLCV with auto-pagination."""
    start_ms = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ms = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        raw = client.klines(symbol, interval, startTime=cursor, endTime=end_ms, limit=1500)
        if not raw:
            break
        all_rows.extend(raw)
        cursor = int(raw[-1][6]) + 1
        if len(raw) < 1500:
            break
        time.sleep(0.15)
    df = pd.DataFrame(all_rows, columns=_ALL_COLS)
    for c in ['open','high','low','close','volume','taker_buy_base','taker_buy_quote']:
        df[c] = df[c].astype(float)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    print(f'{symbol} {interval}: {len(df)} bars  [{df.timestamp.iloc[0]} → {df.timestamp.iloc[-1]}]')
    return df

SYMBOLS = ['BTCUSDT', 'ETHUSDT']
START, END = '2025-12-01', '2026-05-31'

data = {}
for sym in SYMBOLS:
    data[sym] = {}
    for tf in ['1h', '4h']:
        data[sym][tf] = fetch_ohlcv(sym, tf, START, END)

print(f'\nTotal datasets: {sum(len(data[s]) for s in data)}')

## Run Regime Module on All Datasets

In [ ]:
# ── Run RegimeOrchestrator.analyze_series() on each dataset ────────
regime_results = {}

for sym in SYMBOLS:
    regime_results[sym] = {}
    for tf in ['1h', '4h']:
        print(f'\n--- {sym} {tf} ---')
        orch = RegimeOrchestrator.create(sym, tf)
        df = data[sym][tf].copy()
        regime_df = orch.analyze_series(df)
        
        # Attach price for forward return calculations
        regime_df['close'] = df['close'].values[:len(regime_df)]
        regime_df['timestamp'] = df['timestamp'].values[:len(regime_df)]
        regime_results[sym][tf] = regime_df
        
        print(f'  Output columns: {list(regime_df.columns)}')
        print(f'  Regimes found: {regime_df["regime"].value_counts().to_dict()}')
        print(f'  p_trending range: [{regime_df["p_trending"].min():.4f}, {regime_df["p_trending"].max():.4f}]')
        print(f'  vol_percentile range: [{regime_df["vol_percentile"].min():.1f}, {regime_df["vol_percentile"].max():.1f}]')

print('\nAll regime series computed.')

---
## H1: Regime Distribution & Transition Frequency

**Question**: Do we have enough regime transitions per group (especially TREND_BEAR) to calibrate 16 ensemble weights?

**Go criteria**: Each of the 4 proposed groups has ≥30 distinct segments (regime entries) across all datasets.

In [ ]:
# ── H1: Regime distribution, dwell times, transition matrix ────────

# Proposed 4-group mapping
GROUP_MAP = {
    'CLEAN_TREND_BULL': 'TREND_BULL',
    'VOLATILE_TREND_BULL': 'TREND_BULL',
    'CLEAN_TREND_BEAR': 'TREND_BEAR',
    'VOLATILE_TREND_BEAR': 'TREND_BEAR',
    'CLEAN_TREND_FLAT': 'RANGE',    # Flat trends ≈ range behavior
    'VOLATILE_TREND_FLAT': 'CHOPPY', # Volatile flat ≈ choppy
    'QUIET_MR_RANGE': 'RANGE',
    'QUIET_MR_SQUEEZE': 'RANGE',
    'CHOPPY': 'CHOPPY',
}

def compute_regime_stats(regime_series: pd.Series, label: str):
    """Compute distribution, segment count, dwell times."""
    # Bar counts per regime
    counts = regime_series.value_counts()
    pct = (counts / len(regime_series) * 100).round(1)
    
    # Count segments (consecutive runs)
    changes = regime_series != regime_series.shift(1)
    segment_ids = changes.cumsum()
    segments = regime_series.groupby(segment_ids).agg(['first', 'count'])
    segments.columns = ['regime', 'dwell_bars']
    segments = segments.reset_index(drop=True)  # avoid index/column ambiguity
    
    seg_counts = segments['regime'].value_counts()
    dwell_stats = segments.groupby('regime')['dwell_bars'].agg(['mean', 'median', 'min', 'max'])
    
    print(f'\n{"=" * 60}')
    print(f'{label}: {len(regime_series)} bars, {len(segments)} segments, '
          f'{changes.sum()} transitions')
    print(f'{"=" * 60}')
    
    # Per-regime table
    summary = pd.DataFrame({
        'bars': counts,
        'pct': pct,
        'segments': seg_counts,
        'mean_dwell': dwell_stats['mean'].round(1),
        'median_dwell': dwell_stats['median'],
        'min_dwell': dwell_stats['min'],
        'max_dwell': dwell_stats['max'],
    }).fillna(0).sort_values('bars', ascending=False)
    print(summary.to_string())
    
    # 4-group stats
    grouped = regime_series.map(GROUP_MAP)
    g_changes = grouped != grouped.shift(1)
    g_seg_ids = g_changes.cumsum()
    g_segments = grouped.groupby(g_seg_ids).agg(['first', 'count'])
    g_segments.columns = ['group', 'dwell_bars']
    g_segments = g_segments.reset_index(drop=True)
    
    g_seg_counts = g_segments['group'].value_counts()
    g_dwell = g_segments.groupby('group')['dwell_bars'].agg(['mean', 'median'])
    g_bar_counts = grouped.value_counts()
    g_pct = (g_bar_counts / len(grouped) * 100).round(1)
    
    print(f'\n  4-Group Summary:')
    g_summary = pd.DataFrame({
        'bars': g_bar_counts,
        'pct': g_pct,
        'segments': g_seg_counts,
        'mean_dwell': g_dwell['mean'].round(1),
    }).sort_values('bars', ascending=False)
    print(g_summary.to_string())
    
    return segments, g_segments

# Run for all datasets
all_segments = {}
for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        label = f'{sym} {tf}'
        segs, g_segs = compute_regime_stats(regime_results[sym][tf]['regime'], label)
        all_segments[label] = g_segs

# Aggregate across all datasets
print(f'\n{"=" * 60}')
print('AGGREGATE 4-GROUP SEGMENT COUNTS (all sym × tf)')
print(f'{"=" * 60}')
agg_counts = Counter()
for label, g_segs in all_segments.items():
    for grp, cnt in g_segs['group'].value_counts().items():
        agg_counts[grp] += cnt
for grp in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']:
    n = agg_counts.get(grp, 0)
    verdict = 'GO' if n >= 30 else 'LOW'
    print(f'  {grp:15s}: {n:4d} segments  [{verdict}]')

In [ ]:
# ── H1 Compact Summary ──────────────────────────────────────────────
print('=' * 80)
print('H1 COMPACT SUMMARY: Regime Distribution & Segment Counts')
print('=' * 80)

GROUPS = {
    'TREND_BULL': ['CLEAN_TREND_BULL', 'VOLATILE_TREND_BULL'],
    'TREND_BEAR': ['CLEAN_TREND_BEAR', 'VOLATILE_TREND_BEAR'],
    'RANGE': ['QUIET_MR_RANGE', 'QUIET_MR_SQUEEZE', 'CLEAN_TREND_FLAT'],
    'CHOPPY': ['CHOPPY', 'VOLATILE_TREND_FLAT'],
}

labels = [f'{s} {t}' for s in SYMBOLS for t in ['1h', '4h']]
print(f'\n{"Group":<14} ', end='')
for label in labels:
    print(f'{label:>16}', end='')
print()
print('-' * 82)

go_criteria_met = True
for group, regimes in GROUPS.items():
    print(f'{group:<14} ', end='')
    total_segments = 0
    for sym in SYMBOLS:
        for tf in ['1h', '4h']:
            r_df = regime_results[sym][tf]
            in_group = r_df['regime'].isin(regimes)
            segments = (in_group & ~in_group.shift(1, fill_value=False)).sum()
            total_segments += segments
            pct = in_group.mean() * 100
            print(f'{pct:>8.1f}% ({segments:>3})', end='')
    print(f'  total={total_segments}')
    if total_segments < 30:
        go_criteria_met = False
        print(f'  ⚠️  BELOW 30 segments threshold!')

print(f'\nH1 GO criteria (≥30 segments per group): {"PASS ✓" if go_criteria_met else "FAIL ✗"}')

---
## H1b: Transition Matrix (9-state)

Visualize which regimes transition to which — important for understanding regime persistence.

In [ ]:
# ── H1b: Transition matrix heatmap ────────────────────────────────

def compute_transition_matrix(regime_series: pd.Series, regime_list: list) -> np.ndarray:
    """Compute normalized transition matrix."""
    n_r = len(regime_list)
    idx_map = {r: i for i, r in enumerate(regime_list)}
    trans = np.zeros((n_r, n_r))
    prev = regime_series.iloc[0]
    for r in regime_series.iloc[1:]:
        if r != prev:
            i, j = idx_map.get(prev), idx_map.get(r)
            if i is not None and j is not None:
                trans[i, j] += 1
        prev = r
    # Normalize rows
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return trans / row_sums

# Use BTC 1h as primary (most bars)
regime_order = [
    'CLEAN_TREND_BULL', 'CLEAN_TREND_BEAR', 'CLEAN_TREND_FLAT',
    'VOLATILE_TREND_BULL', 'VOLATILE_TREND_BEAR', 'VOLATILE_TREND_FLAT',
    'QUIET_MR_RANGE', 'QUIET_MR_SQUEEZE', 'CHOPPY',
]
short_labels = ['CT_BULL', 'CT_BEAR', 'CT_FLAT', 'VT_BULL', 'VT_BEAR', 'VT_FLAT',
                'QMR_RNG', 'QMR_SQZ', 'CHOPPY']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax_idx, sym in enumerate(SYMBOLS):
    tm = compute_transition_matrix(regime_results[sym]['1h']['regime'], regime_order)
    im = axes[ax_idx].imshow(tm, cmap='YlOrRd', vmin=0, vmax=0.5)
    axes[ax_idx].set_xticks(range(len(short_labels)))
    axes[ax_idx].set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
    axes[ax_idx].set_yticks(range(len(short_labels)))
    axes[ax_idx].set_yticklabels(short_labels, fontsize=8)
    axes[ax_idx].set_title(f'{sym} 1h — Transition Probabilities')
    
    for i in range(len(regime_order)):
        for j in range(len(regime_order)):
            if tm[i, j] > 0.01:
                axes[ax_idx].text(j, i, f'{tm[i,j]:.2f}', ha='center', va='center',
                                 fontsize=7, color='black' if tm[i,j] > 0.25 else 'white')

plt.colorbar(im, ax=axes, shrink=0.8, label='Transition Probability')
plt.suptitle('H1b: 9-State Regime Transition Matrix', fontsize=14)
plt.tight_layout()
plt.show()

---
## H2: Conditional Returns Differ by Regime

**Question**: Are forward returns statistically different across the 4 regime groups?

**Method**: Welch t-test (unequal variance) between each pair of groups.

**Go criteria**: At least 2 group pairs with p < 0.05 and |mean_diff| > 0.1% per bar.

In [ ]:
# ── H2: Conditional forward returns by regime group ────────────────

HORIZONS = [1, 6, 12, 24]  # bars forward

def analyze_conditional_returns(regime_df: pd.DataFrame, label: str):
    """Compute forward returns by regime group, run Welch t-tests."""
    df = regime_df.copy()
    df['group'] = df['regime'].map(GROUP_MAP)
    
    # Compute forward returns
    for h in HORIZONS:
        df[f'fwd_ret_{h}'] = df['close'].pct_change(h).shift(-h)
    
    groups = ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']
    
    print(f'\n{"=" * 70}')
    print(f'{label}: Conditional Mean Forward Returns (bps per bar)')
    print(f'{"=" * 70}')
    
    # Mean returns table
    for h in HORIZONS:
        col = f'fwd_ret_{h}'
        row_data = {}
        for grp in groups:
            mask = df['group'] == grp
            vals = df.loc[mask, col].dropna()
            mean_bps = vals.mean() * 10000 / h  # bps per bar
            row_data[grp] = f'{mean_bps:+.2f} (n={len(vals)})'
        print(f'  h={h:2d}: ' + '  |  '.join(f'{g}: {v}' for g, v in row_data.items()))
    
    # Welch t-tests for h=12 (primary horizon)
    h = 12
    col = f'fwd_ret_{h}'
    print(f'\n  Welch t-tests (h={h}):')
    results = []
    for i, g1 in enumerate(groups):
        for g2 in groups[i+1:]:
            v1 = df.loc[df['group'] == g1, col].dropna().values
            v2 = df.loc[df['group'] == g2, col].dropna().values
            if len(v1) >= 10 and len(v2) >= 10:
                t_stat, p_val = sp_stats.ttest_ind(v1, v2, equal_var=False)
                diff_bps = (v1.mean() - v2.mean()) * 10000
                sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.1 else ''
                results.append((g1, g2, diff_bps, t_stat, p_val, sig))
                print(f'    {g1:12s} vs {g2:12s}: diff={diff_bps:+.2f}bps  t={t_stat:+.3f}  p={p_val:.4f} {sig}')
    
    # Overall ANOVA (Kruskal-Wallis, non-parametric)
    group_vals = [df.loc[df['group'] == g, col].dropna().values for g in groups]
    group_vals = [v for v in group_vals if len(v) >= 5]
    if len(group_vals) >= 2:
        kw_stat, kw_p = sp_stats.kruskal(*group_vals)
        print(f'\n  Kruskal-Wallis (all groups): H={kw_stat:.3f}, p={kw_p:.4f} '
              f'{"*** SIGNIFICANT" if kw_p < 0.05 else "not significant"}')
    
    return df

# Run for all datasets
enriched = {}
for sym in SYMBOLS:
    enriched[sym] = {}
    for tf in ['1h', '4h']:
        enriched[sym][tf] = analyze_conditional_returns(
            regime_results[sym][tf], f'{sym} {tf}'
        )

### H2b: Return Distribution Plots by Regime Group

In [ ]:
# ── H2b: Box plots of forward returns by regime group ─────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
groups = ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']
colors = {'TREND_BULL': '#00ff88', 'TREND_BEAR': '#ff4444', 'RANGE': '#4488ff', 'CHOPPY': '#ffaa00'}

for ax_idx, (sym, tf) in enumerate([('BTCUSDT','1h'), ('ETHUSDT','1h'), ('BTCUSDT','4h'), ('ETHUSDT','4h')]):
    ax = axes[ax_idx // 2][ax_idx % 2]
    df = enriched[sym][tf]
    col = 'fwd_ret_12'
    
    data_to_plot = []
    labels_to_plot = []
    for grp in groups:
        vals = df.loc[df['group'] == grp, col].dropna().values * 100  # percent
        if len(vals) > 0:
            data_to_plot.append(vals)
            labels_to_plot.append(f'{grp}\n(n={len(vals)})')
    
    bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True,
                    showfliers=False, widths=0.6)
    for patch, grp in zip(bp['boxes'], groups):
        patch.set_facecolor(colors.get(grp, 'gray'))
        patch.set_alpha(0.5)
    
    ax.axhline(0, color='white', linewidth=0.5, alpha=0.5)
    ax.set_ylabel('12-bar Forward Return (%)')
    ax.set_title(f'{sym} {tf}')

plt.suptitle('H2b: Forward Return Distributions by Regime Group (h=12)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── H2 Compact Summary ──────────────────────────────────────────────
print('=' * 80)
print('H2 COMPACT: Conditional Returns by Regime Group (h=12)')
print('=' * 80)

GROUP_MAP = {
    'CLEAN_TREND_BULL': 'TREND_BULL', 'VOLATILE_TREND_BULL': 'TREND_BULL',
    'CLEAN_TREND_BEAR': 'TREND_BEAR', 'VOLATILE_TREND_BEAR': 'TREND_BEAR',
    'CLEAN_TREND_FLAT': 'RANGE', 'QUIET_MR_RANGE': 'RANGE', 'QUIET_MR_SQUEEZE': 'RANGE',
    'VOLATILE_TREND_FLAT': 'CHOPPY', 'CHOPPY': 'CHOPPY',
}

print(f'{"Sym/TF":<14} {"TREND_BULL":>12} {"TREND_BEAR":>12} {"RANGE":>12} {"CHOPPY":>12} {"Welch p":>10}')
print('-' * 74)

sig_pairs = 0
total_pairs = 0
for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        df = regime_results[sym][tf].copy()
        df['fwd_12'] = df['close'].pct_change(12).shift(-12) * 100
        df['group'] = df['regime'].map(GROUP_MAP)
        
        means = []
        for g in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']:
            subset = df[df['group'] == g]['fwd_12'].dropna()
            means.append(f'{subset.mean():>+11.3f}%' if len(subset) > 5 else f'{"n/a":>12}')
        
        # Welch t-test between most extreme pair
        groups_data = {g: df[df['group'] == g]['fwd_12'].dropna() for g in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']}
        min_p = 1.0
        for g1 in groups_data:
            for g2 in groups_data:
                if g1 >= g2 or len(groups_data[g1]) < 5 or len(groups_data[g2]) < 5:
                    continue
                total_pairs += 1
                _, p = sp_stats.ttest_ind(groups_data[g1], groups_data[g2], equal_var=False)
                if p < 0.05:
                    sig_pairs += 1
                min_p = min(min_p, p)
        
        print(f'{sym}/{tf:<4} {"  ".join(means)}  p={min_p:.4f}')

print(f'\nSignificant pairs (p<0.05): {sig_pairs}/{total_pairs}')
print(f'H2 GO criteria (≥2 pairs p<0.05 & |mean_diff|>0.1%): {"PASS ✓" if sig_pairs >= 2 else "FAIL ✗"}')

---
## H3: Posterior Continuity

**Question**: Are `p_trending` and `vol_percentile` continuous enough for soft blending?

**Go criteria**: Both posteriors use >10 distinct values and autocorrelation(lag=1) > 0.8.

In [ ]:
# ── H3: Posterior continuity analysis ──────────────────────────────

print('H3: Posterior Continuity Analysis')
print('=' * 70)

for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        rdf = regime_results[sym][tf]
        label = f'{sym} {tf}'
        
        for col_name, display in [('p_trending', 'P(trending)'), ('vol_percentile', 'Vol Percentile')]:
            vals = rdf[col_name].dropna()
            n_unique = vals.nunique()
            ac1 = vals.autocorr(lag=1)
            ac5 = vals.autocorr(lag=5)
            is_binary = n_unique <= 5
            
            verdict = 'CONTINUOUS' if n_unique > 10 and ac1 > 0.8 else 'BINARY/CHOPPY'
            
            print(f'  {label} {display:18s}: unique={n_unique:5d}  '
                  f'AC(1)={ac1:.4f}  AC(5)={ac5:.4f}  '
                  f'range=[{vals.min():.4f}, {vals.max():.4f}]  '
                  f'[{verdict}]')
        print()

In [ ]:
# ── H3b: Time-series plots of posteriors ──────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)

# BTC 1h as primary view
rdf = regime_results['BTCUSDT']['1h']
x = range(len(rdf))

# Panel 1: Price
axes[0].plot(x, rdf['close'], color='white', linewidth=0.8)
axes[0].set_ylabel('Price')
axes[0].set_title('BTCUSDT 1h — Regime Posteriors Over Time')

# Panel 2: p_trending
axes[1].plot(x, rdf['p_trending'], color='#00ccff', linewidth=0.8)
axes[1].axhline(0.5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].set_ylabel('P(trending)')
axes[1].set_ylim(-0.05, 1.05)

# Panel 3: vol_percentile
axes[2].plot(x, rdf['vol_percentile'], color='#ffaa00', linewidth=0.8)
axes[2].axhline(70, color='red', linewidth=0.5, linestyle='--', alpha=0.5, label='HIGH_VOL threshold')
axes[2].axhline(30, color='cyan', linewidth=0.5, linestyle='--', alpha=0.5, label='SQUEEZE threshold')
axes[2].set_ylabel('Vol Percentile')
axes[2].legend(fontsize=8)

# Panel 4: Regime colored bands
regime_colors = {
    'CLEAN_TREND_BULL': '#00ff88', 'CLEAN_TREND_BEAR': '#ff4444',
    'CLEAN_TREND_FLAT': '#888888', 'VOLATILE_TREND_BULL': '#00aa55',
    'VOLATILE_TREND_BEAR': '#cc0000', 'VOLATILE_TREND_FLAT': '#555555',
    'QUIET_MR_RANGE': '#4488ff', 'QUIET_MR_SQUEEZE': '#6644ff',
    'CHOPPY': '#ffaa00',
}
regimes = rdf['regime'].values
prev_r = regimes[0]
start_i = 0
for i in range(1, len(regimes)):
    if regimes[i] != prev_r or i == len(regimes) - 1:
        axes[3].axvspan(start_i, i, alpha=0.4, color=regime_colors.get(prev_r, 'gray'))
        prev_r = regimes[i]
        start_i = i
axes[3].set_ylabel('Regime')
axes[3].set_xlabel('Bar Index')

# Legend for regime colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.5, label=r.replace('_', ' '))
                   for r, c in regime_colors.items()]
axes[3].legend(handles=legend_elements, loc='upper left', fontsize=6, ncol=3)

plt.tight_layout()
plt.show()

### H3c: Posterior Histogram — Is p_trending Bimodal?

In [ ]:
# ── H3c: Histogram of p_trending — check if bimodal or continuous ─

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax_idx, (sym, tf) in enumerate([('BTCUSDT','1h'), ('ETHUSDT','1h'), ('BTCUSDT','4h'), ('ETHUSDT','4h')]):
    ax = axes[ax_idx // 2][ax_idx % 2]
    rdf = regime_results[sym][tf]
    
    ax.hist(rdf['p_trending'].dropna(), bins=50, alpha=0.7, color='#00ccff', edgecolor='black')
    ax.axvline(0.5, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('P(trending)')
    ax.set_ylabel('Count')
    ax.set_title(f'{sym} {tf} — p_trending distribution')
    
    # Check bimodality: Hartigan's dip test approximation
    vals = rdf['p_trending'].dropna().values
    below = (vals < 0.3).sum()
    above = (vals > 0.7).sum()
    middle = ((vals >= 0.3) & (vals <= 0.7)).sum()
    ax.text(0.02, 0.95, f'<0.3: {below} | 0.3-0.7: {middle} | >0.7: {above}',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

plt.suptitle('H3c: p_trending Distribution — Bimodal or Continuous?', fontsize=14)
plt.tight_layout()
plt.show()

---
## H4: 9→4 Grouping Validity

**Question**: Is the architect's TREND_BULL / TREND_BEAR / RANGE / CHOPPY grouping supported by the data?

**Method**: Compare within-group vs between-group variance of conditional returns. If grouping is natural, within-group variance should be much smaller.

In [ ]:
# ── H4: Grouping validity — within vs between variance ────────────

print('H4: 9→4 Grouping Validity — Conditional Return Profiles')
print('=' * 70)

for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        df = enriched[sym][tf]
        label = f'{sym} {tf}'
        col = 'fwd_ret_12'
        
        # Per-regime mean returns (9 regimes)
        regime_means = {}
        for r in ALL_REGIMES:
            vals = df.loc[df['regime'] == r, col].dropna()
            if len(vals) >= 10:
                regime_means[r] = {'mean': vals.mean() * 10000, 'std': vals.std() * 10000, 'n': len(vals)}
        
        print(f'\n{label}: Per-Regime Mean Forward Return (bps, h=12)')
        for r in ALL_REGIMES:
            if r in regime_means:
                m = regime_means[r]
                grp = GROUP_MAP[r]
                print(f'  {r:25s} → {grp:12s}: mean={m["mean"]:+7.2f}bps  std={m["std"]:6.1f}  n={m["n"]}')
        
        # Check: do members of same group have similar means?
        groups = ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']
        print(f'\n  Within-Group Coherence (do sub-regimes agree on sign/magnitude?):')
        for grp in groups:
            members = [r for r in ALL_REGIMES if GROUP_MAP[r] == grp and r in regime_means]
            if len(members) >= 2:
                means = [regime_means[r]['mean'] for r in members]
                spread = max(means) - min(means)
                same_sign = all(m > 0 for m in means) or all(m <= 0 for m in means)
                verdict = 'COHERENT' if same_sign and spread < 20 else 'MIXED' if not same_sign else 'SPREAD'
                print(f'    {grp:12s}: members={members}  means={["{:.1f}".format(m) for m in means]}  '
                      f'spread={spread:.1f}bps  same_sign={same_sign}  [{verdict}]')
            elif len(members) == 1:
                print(f'    {grp:12s}: single member [{members[0]}] — n/a')

In [ ]:
# ── H4b: Heatmap of regime mean returns across sym × tf ───────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax_idx, (sym, tf) in enumerate([('BTCUSDT','1h'), ('ETHUSDT','1h'), ('BTCUSDT','4h'), ('ETHUSDT','4h')]):
    ax = axes[ax_idx // 2][ax_idx % 2]
    df = enriched[sym][tf]
    
    mean_matrix = []
    for r in ALL_REGIMES:
        row = []
        for h in HORIZONS:
            vals = df.loc[df['regime'] == r, f'fwd_ret_{h}'].dropna()
            row.append(vals.mean() * 10000 if len(vals) >= 5 else np.nan)
        mean_matrix.append(row)
    
    mean_arr = np.array(mean_matrix)
    short_labels = ['CT_B', 'CT_Be', 'CT_F', 'VT_B', 'VT_Be', 'VT_F', 'QR', 'QS', 'CH']
    
    im = ax.imshow(mean_arr, cmap='RdYlGn', vmin=-30, vmax=30, aspect='auto')
    ax.set_xticks(range(len(HORIZONS)))
    ax.set_xticklabels([f'h={h}' for h in HORIZONS])
    ax.set_yticks(range(len(short_labels)))
    ax.set_yticklabels(short_labels, fontsize=8)
    ax.set_title(f'{sym} {tf}')
    
    for i in range(len(short_labels)):
        for j in range(len(HORIZONS)):
            if not np.isnan(mean_arr[i, j]):
                ax.text(j, i, f'{mean_arr[i,j]:+.0f}', ha='center', va='center', fontsize=7,
                        color='black' if abs(mean_arr[i,j]) < 15 else 'white')

plt.colorbar(im, ax=axes, shrink=0.8, label='Mean Fwd Return (bps)')
plt.suptitle('H4b: Mean Forward Returns by Regime × Horizon', fontsize=14)
plt.tight_layout()
plt.show()

---
## H5: MTF Alignment Value

**Question**: Does HTF (4h) confirmation improve 1h forward returns?

**Method**: Run `analyze_series_mtf()`, compare returns when HTF confirms vs conflicts.

In [ ]:
# ── H5: MTF fusion — does HTF confirmation add value? ─────────────

print('H5: MTF Alignment Value')
print('=' * 70)

mtf_results = {}

for sym in SYMBOLS:
    print(f'\n--- {sym} ---')
    orch = RegimeOrchestrator.create(sym, '1h')
    df_1h = data[sym]['1h'].copy()
    df_4h = data[sym]['4h'].copy()
    
    mtf_df = orch.analyze_series_mtf(df_1h, df_4h, '4h')
    mtf_df['close'] = df_1h['close'].values[:len(mtf_df)]
    
    print(f'  MTF output columns: {list(mtf_df.columns)}')
    print(f'  MTF output length: {len(mtf_df)}')
    mtf_results[sym] = mtf_df

In [ ]:
# ── H5b: Compare returns by MTF alignment status ──────────────────

for sym in SYMBOLS:
    mtf_df = mtf_results[sym].copy()
    
    # Compute forward returns
    for h in HORIZONS:
        mtf_df[f'fwd_ret_{h}'] = mtf_df['close'].pct_change(h).shift(-h)
    
    # Check available MTF columns
    mtf_cols = [c for c in mtf_df.columns if 'mtf' in c.lower() or 'align' in c.lower() or 'fused' in c.lower()]
    print(f'\n{sym} MTF-related columns: {mtf_cols}')
    
    # Try to find alignment/confirmation column
    # The MTFFusion should produce position_scale adjustments
    if 'position_scale' in mtf_df.columns:
        # Compare MTF position_scale vs single-TF position_scale
        single_ps = regime_results[sym]['1h']['position_scale'].values[:len(mtf_df)]
        mtf_ps = mtf_df['position_scale'].values
        
        # Infer alignment: MTF boosted (>1x single) vs penalized (<1x single)
        ratio = np.where(np.abs(single_ps) > 0.01, mtf_ps / single_ps, 1.0)
        mtf_df['mtf_ratio'] = ratio
        mtf_df['mtf_status'] = np.where(ratio > 1.05, 'CONFIRMING',
                                        np.where(ratio < 0.95, 'CONFLICTING', 'NEUTRAL'))
        
        status_counts = mtf_df['mtf_status'].value_counts()
        print(f'  MTF status distribution: {status_counts.to_dict()}')
        
        # Compare forward returns by MTF status
        print(f'\n  Forward returns by MTF alignment (bps, h=12):')
        for status in ['CONFIRMING', 'NEUTRAL', 'CONFLICTING']:
            mask = mtf_df['mtf_status'] == status
            if mask.sum() >= 10:
                vals = mtf_df.loc[mask, 'fwd_ret_12'].dropna()
                abs_ret = vals.abs().mean() * 10000
                print(f'    {status:12s}: mean={vals.mean()*10000:+.2f}bps  '
                      f'|mean|={abs_ret:.2f}bps  std={vals.std()*10000:.1f}  n={len(vals)}')
        
        # T-test: CONFIRMING vs CONFLICTING
        conf_vals = mtf_df.loc[mtf_df['mtf_status'] == 'CONFIRMING', 'fwd_ret_12'].dropna().values
        confl_vals = mtf_df.loc[mtf_df['mtf_status'] == 'CONFLICTING', 'fwd_ret_12'].dropna().values
        if len(conf_vals) >= 10 and len(confl_vals) >= 10:
            t_stat, p_val = sp_stats.ttest_ind(np.abs(conf_vals), np.abs(confl_vals), equal_var=False)
            print(f'\n  Welch t-test |CONFIRMING| vs |CONFLICTING|: t={t_stat:.3f}, p={p_val:.4f}')
    else:
        print(f'  position_scale not found in MTF output. Available columns: {list(mtf_df.columns)}')

---
## Synthesis: Go / No-Go for Regime-Weighted Ensemble

In [ ]:
# ── Final Synthesis ────────────────────────────────────────────────

print('\n' + '=' * 70)
print('SYNTHESIS: Regime Foundation Validation')
print('=' * 70)

# H1: Aggregate segment counts per group
h1_verdict = {}
agg_counts = Counter()
for label, g_segs in all_segments.items():
    for grp, cnt in g_segs['group'].value_counts().items():
        agg_counts[grp] += cnt

print('\nH1: Regime Distribution & Transitions')
h1_pass = True
for grp in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']:
    n = agg_counts.get(grp, 0)
    ok = n >= 30
    h1_pass = h1_pass and ok
    print(f'  {grp:15s}: {n:4d} segments  [{"GO" if ok else "INSUFFICIENT"}]')
print(f'  → H1 Verdict: {"GO" if h1_pass else "NO-GO"}')

# H2: Count significant t-test pairs
print('\nH2: Conditional Returns Differ by Regime')
h2_sig_pairs = 0
for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        df = enriched[sym][tf]
        groups = ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']
        for i, g1 in enumerate(groups):
            for g2 in groups[i+1:]:
                v1 = df.loc[df['group'] == g1, 'fwd_ret_12'].dropna().values
                v2 = df.loc[df['group'] == g2, 'fwd_ret_12'].dropna().values
                if len(v1) >= 10 and len(v2) >= 10:
                    _, p_val = sp_stats.ttest_ind(v1, v2, equal_var=False)
                    if p_val < 0.05:
                        h2_sig_pairs += 1
h2_pass = h2_sig_pairs >= 2
print(f'  Significant pairs (p<0.05): {h2_sig_pairs}')
print(f'  → H2 Verdict: {"GO" if h2_pass else "NO-GO (returns not differentiated)"}')

# H3: Posterior continuity
print('\nH3: Posterior Continuity')
h3_pass = True
for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        rdf = regime_results[sym][tf]
        for col_name in ['p_trending', 'vol_percentile']:
            vals = rdf[col_name].dropna()
            n_unique = vals.nunique()
            ac1 = vals.autocorr(lag=1)
            if n_unique <= 10 or ac1 < 0.8:
                h3_pass = False
                print(f'  WARN: {sym} {tf} {col_name}: unique={n_unique}, AC(1)={ac1:.4f}')
if h3_pass:
    print(f'  All posteriors continuous and autocorrelated.')
print(f'  → H3 Verdict: {"GO" if h3_pass else "CAUTION — some posteriors may be binary"}')

# H4: Grouping coherence — simple check
print('\nH4: 9→4 Grouping Validity')
h4_coherent = 0
h4_total = 0
for sym in SYMBOLS:
    for tf in ['1h', '4h']:
        df = enriched[sym][tf]
        for grp in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']:
            members = [r for r in ALL_REGIMES if GROUP_MAP[r] == grp]
            means = []
            for r in members:
                vals = df.loc[df['regime'] == r, 'fwd_ret_12'].dropna()
                if len(vals) >= 10:
                    means.append(vals.mean())
            if len(means) >= 2:
                h4_total += 1
                if all(m > 0 for m in means) or all(m <= 0 for m in means):
                    h4_coherent += 1
h4_pass = h4_total == 0 or (h4_coherent / h4_total >= 0.6)
print(f'  Coherent groups (same sign): {h4_coherent}/{h4_total}')
print(f'  → H4 Verdict: {"GO" if h4_pass else "REVIEW GROUPING"}')

# Overall
overall = h1_pass and h2_pass and h3_pass and h4_pass
print(f'\n{"=" * 70}')
print(f'OVERALL: {"GO — Proceed with Regime-Weighted Ensemble" if overall else "REVIEW NEEDED — see warnings above"}')
print(f'{"=" * 70}')